# 🔥 Thermal DIP: Google Colab GPU Backend & Training Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/likhith-naik01/DIP_MP/blob/main/ml/notebooks/Thermal_DIP_Colab_GPU_Backend.ipynb)

This notebook runs the **FastAPI Backend and PyTorch Deep Image Prior (DIP)** worker directly on Google Colab's free GPU (NVIDIA T4 / V100 / A100).

It creates a public HTTPS tunnel (via Cloudflare Tunnel or ngrok) so your local React frontend can connect directly to Colab for **15-30 second GPU-accelerated thermal image restorations** instead of waiting minutes on CPU.

### Step 1: Verify GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. In Colab, go to Runtime -> Change runtime type -> Select T4 GPU.")

### Step 2: Clone Repository and Install Dependencies

In [ ]:
# Clone or pull latest code
import os
if not os.path.exists("DIP_MP"):
    !git clone https://github.com/likhith-naik01/DIP_MP.git
else:
    %cd DIP_MP
    !git pull
    %cd ..

# Install backend and ml dependencies
!pip install -q fastapi uvicorn sse-starlette python-multipart pydantic-settings sqlalchemy scikit-image
%cd DIP_MP/ml
!pip install -q -e .
%cd ../backend

### Step 3: Option A - Run FastAPI Backend with Cloudflare Tunnel (No Sign-up Required!)
This runs the FastAPI server in the background and opens an encrypted public HTTPS URL (`https://*.trycloudflare.com`) that you can paste directly into the React Frontend.

In [ ]:
# Download cloudflared binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# Start FastAPI server in background
import subprocess
import time
import re

print("Starting FastAPI GPU Server on port 8000...")
backend_proc = subprocess.Popen([
    "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(3)

# Start cloudflared tunnel
print("Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen([
    "./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Find public URL
tunnel_url = None
start_t = time.time()
while time.time() - start_t < 30:
    line = tunnel_proc.stderr.readline().decode('utf-8')
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

print("\n===================================================================")
if tunnel_url:
    print(f"🚀 SUCCESS! Colab GPU Backend is live at:")
    print(f"👉 {tunnel_url}")
    print("\nCopy this URL and paste it into the React Frontend Settings panel!")
else:
    print("Tunnel starting, check logs or alternative ngrok cell below.")
print("===================================================================")

### Step 4: Option B - Alternative Tunnel via pyngrok
If you prefer ngrok, paste your free ngrok token below.

In [ ]:
# !pip install -q pyngrok
# from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")
# public_url = ngrok.connect(8000).public_url
# print(f"Ngrok URL: {public_url}")

### Step 5: Option C - Standalone Benchmark in Colab
Run benchmark script on GPU and output CSV metrics (U-Net DIP vs Conv-DIP vs Mean filter).

In [ ]:
%cd /content/DIP_MP/ml
# Create sample thermal simulation image if needed
from dip_core.pipeline.preprocess import load_image
import numpy as np
from PIL import Image

# Run benchmark on synthetic sample or custom image
!python scripts/benchmark.py --reference ../storage/sample_images/thermal_circuit_board.png --task denoise --iterations 1500